# Build And Export PM2.5 Dataset Bundle

Run this notebook top-to-bottom to collect data files and package them into a downloadable zip.

Outputs are written to `pm25_delhi_bundle/`:
- `stations_urban.csv`
- `openaq_pm25.csv`
- `era5_delhi.nc`
- `era5_meteo.csv`
- `stations_elevation.csv`
- `data_manifest.json`
- `pm25_delhi_bundle.zip`

Requirements:
- OpenAQ API key in environment variable `OPENAQ_KEY`
- CDS API account configured for ERA5 (`~/.cdsapirc`) and license accepted

In [ ]:
from pathlib import Path
import json
import math
import os
import zipfile
from datetime import datetime

import numpy as np
import pandas as pd
import requests

BUNDLE_DIR = Path("pm25_delhi_bundle")
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

# Minimal refresh controls.
REFRESH_ALL = False
REFRESH_TARGETS = {
    "urban": False,
    "openaq": False,
    "era5": False,
    "elevation": False,
}
MIN_OPENAQ_STATIONS = 2

DELHI_BBOX = {
    "lat_min": 28.3,
    "lat_max": 28.9,
    "lon_min": 76.8,
    "lon_max": 77.5,
}

STATIONS = [
    ("STATION_00", 28.63, 77.21),
    ("STATION_01", 28.59, 77.07),
    ("STATION_02", 28.67, 77.12),
    ("STATION_03", 28.55, 77.19),
    ("STATION_04", 28.62, 77.28),
    ("STATION_05", 28.53, 77.24),
    ("STATION_06", 28.65, 77.15),
    ("STATION_07", 28.57, 77.17),
    ("STATION_08", 28.71, 77.17),
    ("STATION_09", 28.70, 77.16),
]

# Expand to 40 points deterministically around anchor stations.
if len(STATIONS) < 40:
    for i in range(40 - len(STATIONS)):
        sid, lat, lon = STATIONS[i % len(STATIONS)]
        lat2 = min(max(lat + ((i % 5) - 2) * 0.01, DELHI_BBOX["lat_min"]), DELHI_BBOX["lat_max"])
        lon2 = min(max(lon + (((i // 5) % 5) - 2) * 0.01, DELHI_BBOX["lon_min"]), DELHI_BBOX["lon_max"])
        STATIONS.append((f"STATION_{len(STATIONS):02d}", float(lat2), float(lon2)))

stations_df = pd.DataFrame(STATIONS, columns=["station_id", "lat", "lon"])
stations_df["source"] = "CPCB"
stations_df = stations_df[["lat", "lon", "source", "station_id"]]

print("Bundle directory:", BUNDLE_DIR.resolve())
print("Stations:", len(stations_df))
print("REFRESH_ALL:", REFRESH_ALL)
print("REFRESH_TARGETS:", REFRESH_TARGETS)
print("MIN_OPENAQ_STATIONS:", MIN_OPENAQ_STATIONS)

In [ ]:
def log(msg: str) -> None:
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    p1 = math.radians(float(lat1))
    p2 = math.radians(float(lat2))
    dp = math.radians(float(lat2) - float(lat1))
    dl = math.radians(float(lon2) - float(lon1))
    a = math.sin(dp / 2.0) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2.0) ** 2
    return 2.0 * r * math.asin(math.sqrt(a))


def update_manifest(bundle_dir: Path):
    manifest_path = bundle_dir / "data_manifest.json"
    old = {}
    if manifest_path.exists():
        old = json.loads(manifest_path.read_text(encoding="utf-8"))

    files = {}
    for name in ["stations_urban.csv", "openaq_pm25.csv", "era5_delhi.nc", "era5_meteo.csv", "stations_elevation.csv"]:
        p = bundle_dir / name
        if p.exists():
            rows = "N/A"
            if p.suffix.lower() == ".csv":
                rows = int(pd.read_csv(p).shape[0])
            prev = old.get("files", {}).get(name, {})
            files[name] = {
                "path": prev.get("path", str(p.resolve())),
                "status": prev.get("status", "REAL"),
                "size_bytes": int(p.stat().st_size),
                "rows": rows,
            }

    manifest = {
        "created_at": str(datetime.now()),
        "files": files,
    }
    manifest_path.write_text(json.dumps(manifest, indent=4), encoding="utf-8")
    log(f"Updated manifest: {manifest_path}")

In [ ]:
# 1) Station urban context (OSMnx when available, otherwise fixed baseline values)
urban_path = BUNDLE_DIR / "stations_urban.csv"

refresh_urban = REFRESH_ALL or REFRESH_TARGETS.get("urban", False)

if urban_path.exists() and not refresh_urban:
    stations_urban = pd.read_csv(urban_path)
    log("Using existing stations_urban.csv")
else:
    try:
        import osmnx as ox

        rows = []
        area_km2 = math.pi * 0.25
        for _, s in stations_df.iterrows():
            lat, lon = float(s["lat"]), float(s["lon"])
            road_density = 45.0
            building_density = 120.0
            try:
                g = ox.graph_from_point((lat, lon), dist=500, network_type="drive")
                road_density = len(g.edges()) / area_km2
                b = ox.features_from_point((lat, lon), tags={"building": True}, dist=500)
                building_density = len(b) / area_km2
            except Exception:
                pass
            rows.append({
                "lat": lat,
                "lon": lon,
                "source": s["source"],
                "station_id": s["station_id"],
                "building_density": float(building_density),
                "road_density": float(road_density),
            })
        stations_urban = pd.DataFrame(rows)
    except Exception:
        stations_urban = stations_df.copy()
        stations_urban["building_density"] = 120.0
        stations_urban["road_density"] = 45.0

    stations_urban.to_csv(urban_path, index=False)
    log("Saved stations_urban.csv")

stations_urban.head()

In [ ]:
# 2) OpenAQ PM2.5 collection and nearest-station aggregation
openaq_path = BUNDLE_DIR / "openaq_pm25.csv"

refresh_openaq = REFRESH_ALL or REFRESH_TARGETS.get("openaq", False)

if openaq_path.exists() and not refresh_openaq:
    openaq_df = pd.read_csv(openaq_path)
    unique_st = openaq_df["station_id"].nunique() if "station_id" in openaq_df.columns else 0
    if unique_st >= MIN_OPENAQ_STATIONS:
        log(f"Using existing openaq_pm25.csv (stations={unique_st})")
    else:
        log(f"Existing openaq_pm25.csv has only {unique_st} station(s); rebuilding OpenAQ file")
        refresh_openaq = True

if (not openaq_path.exists()) or refresh_openaq:
    api_key = os.getenv("OPENAQ_KEY", "").strip()
    if not api_key:
        raise RuntimeError("Set OPENAQ_KEY in your environment before running this cell.")

    headers = {"X-API-Key": api_key}
    base_url = "https://api.openaq.org/v3"

    loc_resp = requests.get(
        f"{base_url}/locations",
        params={
            "coordinates": "28.6139,77.2090",
            "radius": 100000,
            "limit": 100,
            "parameters": "pm25",
        },
        headers=headers,
        timeout=60,
    )
    loc_resp.raise_for_status()
    locations = loc_resp.json().get("results", [])
    if not locations:
        raise RuntimeError("OpenAQ returned no locations.")

    rows = []
    for loc in locations:
        loc_id = loc.get("id")
        coords = loc.get("coordinates", {}) if isinstance(loc.get("coordinates"), dict) else {}
        lat = coords.get("latitude", loc.get("latitude"))
        lon = coords.get("longitude", loc.get("longitude"))
        if loc_id is None or lat is None or lon is None:
            continue

        m_resp = requests.get(
            f"{base_url}/locations/{loc_id}/measurements",
            params={
                "parameters": "pm25",
                "date_from": "2023-01-01",
                "date_to": "2024-12-31",
                "limit": 1000,
            },
            headers=headers,
            timeout=60,
        )
        if m_resp.status_code != 200:
            continue
        measurements = m_resp.json().get("results", [])

        for m in measurements:
            value = m.get("value")
            if value is None:
                continue
            raw_date = None
            if isinstance(m.get("date"), dict):
                raw_date = m["date"].get("utc") or m["date"].get("local")
            raw_date = raw_date or m.get("datetimeUtc") or m.get("datetimeLocal")
            ts = pd.to_datetime(raw_date, errors="coerce", utc=True)
            if pd.isna(ts):
                continue
            rows.append({"lat": float(lat), "lon": float(lon), "date": ts.tz_convert(None).date(), "pm25": float(value)})

    if not rows:
        raise RuntimeError("OpenAQ returned no measurement rows.")

    raw = pd.DataFrame(rows)

    station_coords = stations_urban[["station_id", "lat", "lon"]].drop_duplicates().reset_index(drop=True)
    mapped = []
    for _, r in raw.iterrows():
        d = station_coords.apply(lambda s: haversine_km(r["lat"], r["lon"], s["lat"], s["lon"]), axis=1)
        mapped.append(station_coords.loc[int(d.idxmin()), "station_id"])

    raw["station_id"] = mapped
    openaq_df = raw.groupby(["station_id", "date"], as_index=False)["pm25"].mean().sort_values(["station_id", "date"])

    unique_st = openaq_df["station_id"].nunique()
    if unique_st < MIN_OPENAQ_STATIONS:
        raise RuntimeError(
            f"OpenAQ rebuild resulted in only {unique_st} station(s). Increase radius/limit or check source coverage."
        )

    openaq_df.to_csv(openaq_path, index=False)
    log("Saved openaq_pm25.csv")

print("OpenAQ rows:", len(openaq_df), "unique station_id:", openaq_df["station_id"].nunique())

In [ ]:
# 3) ERA5-Land retrieval and station extraction
era5_nc_path = BUNDLE_DIR / "era5_delhi.nc"
era5_csv_path = BUNDLE_DIR / "era5_meteo.csv"

refresh_era5 = REFRESH_ALL or REFRESH_TARGETS.get("era5", False)

if era5_nc_path.exists() and era5_csv_path.exists() and not refresh_era5:
    era5_df = pd.read_csv(era5_csv_path)
    log("Using existing ERA5 files")
else:
    import cdsapi
    import xarray as xr

    c = cdsapi.Client()
    c.retrieve(
        "reanalysis-era5-land",
        {
            "variable": [
                "2m_temperature",
                "10m_u_component_of_wind",
                "10m_v_component_of_wind",
                "surface_pressure",
                "2m_dewpoint_temperature",
                "total_precipitation",
            ],
            "product_type": "reanalysis",
            "year": ["2023", "2024"],
            "month": [f"{m:02d}" for m in range(1, 13)],
            "day": [f"{d:02d}" for d in range(1, 32)],
            "time": ["12:00"],
            "area": [DELHI_BBOX["lat_max"], DELHI_BBOX["lon_min"], DELHI_BBOX["lat_min"], DELHI_BBOX["lon_max"]],
            "format": "netcdf",
        },
        str(era5_nc_path),
    )

    ds = xr.open_dataset(era5_nc_path)
    lat_name = "latitude" if "latitude" in ds.coords else "lat"
    lon_name = "longitude" if "longitude" in ds.coords else "lon"

    rows = []
    for _, s in stations_urban.iterrows():
        point = ds.sel({lat_name: float(s["lat"]), lon_name: float(s["lon"])}, method="nearest")
        pt = point.to_dataframe().reset_index()
        pt["station_id"] = s["station_id"]
        rows.append(pt)

    era5_df = pd.concat(rows, ignore_index=True)
    era5_df.to_csv(era5_csv_path, index=False)
    log("Saved ERA5 files")

print("ERA5 rows:", len(era5_df), "unique station_id:", era5_df["station_id"].nunique())

In [ ]:
# 4) Elevation extraction from SRTM
srtm_path = BUNDLE_DIR / "stations_elevation.csv"

refresh_elevation = REFRESH_ALL or REFRESH_TARGETS.get("elevation", False)

if srtm_path.exists() and not refresh_elevation:
    elev_df = pd.read_csv(srtm_path)
    log("Using existing stations_elevation.csv")
else:
    import rasterio

    srtm_url = "https://srtm.csi.cgiar.org/wp-content/uploads/files/srtm_5x5/TIFF/srtm_57_07.zip"
    zip_path = BUNDLE_DIR / "srtm_57_07.zip"
    extract_dir = BUNDLE_DIR / "srtm_data"
    extract_dir.mkdir(exist_ok=True)

    resp = requests.get(srtm_url, timeout=120)
    resp.raise_for_status()
    zip_path.write_bytes(resp.content)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)

    tif_files = list(extract_dir.glob("*.tif")) + list(extract_dir.glob("*.TIF"))
    if not tif_files:
        raise RuntimeError("No SRTM TIFF found in archive.")

    nodata_values = {-32768, -9999}
    rows = []
    with rasterio.open(tif_files[0]) as src:
        nodata = src.nodata
        for _, s in stations_urban.iterrows():
            val = float(list(src.sample([(float(s["lon"]), float(s["lat"]))]))[0][0])
            if (nodata is not None and val == nodata) or val in nodata_values or np.isnan(val):
                val = 216.0
            rows.append({
                "lat": s["lat"],
                "lon": s["lon"],
                "source": s["source"],
                "station_id": s["station_id"],
                "elevation": float(val),
            })

    elev_df = pd.DataFrame(rows)
    elev_df.to_csv(srtm_path, index=False)
    log("Saved stations_elevation.csv")

print("Elevation rows:", len(elev_df), "bad rows:", int((pd.to_numeric(elev_df['elevation'], errors='coerce').isna() | elev_df['elevation'].isin([-32768, -9999])).sum()))

In [ ]:
# 5) Finalize manifest and create downloadable zip
update_manifest(BUNDLE_DIR)

zip_path = Path("pm25_delhi_bundle.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(BUNDLE_DIR.glob("*")):
        if p.is_file():
            zf.write(p, arcname=f"pm25_delhi_bundle/{p.name}")

print("Created:", zip_path.resolve())
print("Size (MB):", round(zip_path.stat().st_size / (1024 * 1024), 2))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("If running in VS Code/Jupyter locally, download the zip directly from the workspace.")